# Causal Entropic Forces

Here's how the algorithm works:
1. From the current state, sample `NUM_ROLLOUTS`
2. Estimate the probability of sampling trajectory i `p(traj_i)` using kernel density estimation
3. Define the volume of trajectory i as `vol_i = 1 / p(traj_i)`
4. Calculate the normalized volume of a trajectory `norm_vol_i = vol_i / \sum_i vol_i`
5. Calculate the causal entropic force `2 * T_C / T_R \mean_i f_i * \log norm_vol_i`, where `f_i` is the initial force of trajectory i and the value `T_C / T_R` governs how much more you want to exploit (acting following the entropic force) vs explore (sampling a trajectory)
6. Use the causal entropic force to update the current state

In [1]:
import numpy as np
import scipy

from scipy import linalg

from matplotlib import pyplot as plt
from matplotlib import patches as patches
from matplotlib import cm

import copy

In [2]:
np.random.seed(0)

dtype = np.float32

## Hyperparameters

In [3]:
T_R     = 4e5   # temperature of random agent (K)
T_C     = 2e6   # temperature of causal agent (K)
TAU     = 10.   # simulation time horizon (seconds)
EPSILON = .025  # timesteps (seconds)

In [4]:
NUM_SUBSAMPLES = 5

In [5]:
NUM_ROLLOUTS = 50_000

## Particle-in-a-box environment and dynamics

In [6]:
M     = 1e-21                             # mass (kg)
L     = 400                               # length (meters) 
Q_MIN = np.array([0., 0.  ], dtype=dtype) # minimum box displacements
Q_MAX = np.array([L , L/5.], dtype=dtype) # maximum box displacements

In [7]:
def step_p(p, f):
    p     = f * EPSILON
    p_max = M * np.abs(Q_MAX - Q_MIN) / EPSILON
    new_p = np.sign(p) * np.minimum(p_max, np.abs(p))
    return new_p

In [8]:
def step_q(q, p, new_p):
    new_q = q  +  .5 * EPSILON * (p + new_p) / M
    return new_q

In [9]:
def enforce_elastic_collisions(p, q):
    q  = np.maximum(q, 2 * Q_MIN - q)
    p *= np.sign(q - Q_MIN)
    q  = np.minimum(q, 2 * Q_MAX - q)
    p *= np.sign(Q_MAX - q)
    return p, q

In [10]:
def step_phase_space(p, q, f):
    new_p        = step_p(p, f)
    new_q        = step_q(q, p, new_p)
    new_p, new_q = enforce_elastic_collisions(new_p, new_q)
    return new_p, new_q

## Random rollouts

In [11]:
timesteps = int(TAU / EPSILON)

In [12]:
def generate_fs(num_samples):
    # scale calculation formula from https://math.stackexchange.com/a/1426406
    return np.random.normal( loc   = 0. ,
                             scale = np.sqrt(M * scipy.constants.Boltzmann * T_R) / EPSILON ,
                             size  = (num_samples, 2) ).astype(dtype)

In [13]:
def rollouts(p, q, num_samples):
    """
    paths.shape = (timesteps, num_samples, q_dim)
    """
    ps = np.tile(p[None, :], (num_samples, 1))
    qs = np.tile(q[None, :], (num_samples, 1))

    paths = copy.deepcopy(qs[None, :])

    fs       = generate_fs(num_samples)
    first_fs = copy.deepcopy(fs)
    
    for i in range(timesteps):
        ps, qs = step_phase_space(ps, qs, fs)
        paths  = np.append(paths, qs[None, :], axis=0)
        fs     = generate_fs(num_samples)

    return paths, first_fs

## Causal Entropic Forcing

In [14]:
def subsample(paths):
    # idea and code from Google Gemini to speed up KDE
    sub_indices = np.linspace( 0 ,
                               timesteps - 1 ,
                               NUM_SUBSAMPLES ,
                               dtype=np.int32 ) 
    sub_paths = paths[sub_indices, :, :]
    return sub_paths.transpose((2, 0, 1)).reshape(-1, NUM_ROLLOUTS)

In [15]:
# The following code is modified from 
# [1]: https://github.com/scipy/scipy/blob/main/scipy/stats/_kde.py
# [2]: https://github.com/scipy/scipy/blob/main/scipy/stats/_stats.pyx#L744
# 
# [1] has a copyright notice which I have reproduced below
# 
# -------------------------------------------------------------------------------
#
#  Define classes for (uni/multi)-variate kernel density estimation.
#
#  Currently, only Gaussian kernels are implemented.
#
#  Written by: Robert Kern
#
#  Date: 2004-08-09
#
#  Modified: 2005-02-10 by Robert Kern.
#              Contributed to SciPy
#            2005-10-07 by Robert Kern.
#              Some fixes to match the new scipy_core
#
#  Copyright 2004-2005 by Enthought, Inc.
#
# -------------------------------------------------------------------------------


def gaussian_kde_pdfs(dataset):
    # d is dimensionality of datapoint, m is number of datapoints
    d, m    = dataset.shape
    weights = np.ones(m) / m
    
    cov              = np.cov(dataset, rowvar=1, bias=False, aweights=weights)
    _data_covariance = np.atleast_2d(cov)
    _data_cho_cov    = linalg.cholesky(_data_covariance, lower=True)

    scotts_factor = np.power(m, -1. / (d + 4))
    
    cho_cov = (_data_cho_cov * scotts_factor).astype(np.float32)
    
    # Rescale the data
    dataset_ = linalg.solve_triangular(cho_cov, dataset, lower=True).T

    # just numpy the inner loop
    estimate = np.zeros((m,))
    for j in range(m):
        arg         = np.sum((dataset_ - dataset_[j, :]) ** 2., axis=-1)
        arg         = weights * np.exp(-arg / 2.)
        estimate[j] = arg.sum()
    
    # full numpy version
    # arg      = np.sum((dataset_[:, None] - dataset_[None, :]) ** 2., axis=-1)
    # arg      = weights * np.exp(-arg / 2.)
    # estimate = arg.sum(axis=-1)
    
    norm = np.pow(2 * np.pi, - d / 2.) / np.diag(cho_cov).prod()
    return estimate * norm

In [16]:
def log_vol_fracs(paths, subsampling=True, fuzz=1e-6):
    if subsampling:
        paths = subsample(paths[1:]) # remove first state bc conditional
    paths = (paths - paths.mean()) / paths.std()
    vol   = 1. / (gaussian_kde_pdfs(paths) + fuzz)
    return np.log(vol / vol.sum())
    # return gaussian_kde_pdfs(paths)

In [17]:
def entropic_force(p, q):
    paths, fs = rollouts(p, q, NUM_ROLLOUTS)
    return np.mean( fs * log_vol_fracs(paths)[:, None] , axis=0 )

## Rollouts

In [18]:
p = np.array([0.   , 0.   ])
q = np.array([L/10., L/10.])

path = copy.deepcopy(q[None, :])

for timestep in range(100):
    print(timestep, q)
    f_c  = 2 * T_C / T_R * entropic_force(p, q)
    p, q = step_phase_space(p, q, f_c)
    path = np.append(path, q[None, :], axis=0)

0 [40. 40.]
1 [40.13843816 39.55003043]
2 [40.20393677 38.7015364 ]
3 [40.99402766 38.89112305]
4 [41.4683477  39.32377561]
5 [40.88541162 39.78753806]
6 [40.043084  40.4380264]
7 [38.81763328 40.54478712]
8 [38.97245792 41.16756195]
9 [39.63467948 41.47098103]
10 [39.70777567 40.84143   ]
11 [40.05818757 40.63578404]
12 [40.19806092 41.36958718]
13 [40.46454804 41.2401081 ]
14 [40.28464686 40.59054   ]
15 [39.85279506 42.16896798]


KeyboardInterrupt: 

In [ ]:
fig, ax = plt.subplots(figsize=(13, 13))
ax.scatter(*path[::10].T)
ax.set_xlim(Q_MIN[0], Q_MAX[0])
ax.set_ylim(Q_MIN[1], Q_MAX[1])
ax.set(aspect='equal')
plt.show()